In [ ]:
!jq '. | length' ../data/dataset.json

20000


In [ ]:

!jq ".[$(jq 'keys[]' ../data/dataset.json | shuf -n 1)]" ../data/dataset.json

{
  "id": "pk_01510",
  "author_reputation": 37,
  "version": 1,
  "fork_count": 1,
  "likes": 15,
  "upvotes": 15,
  "downvotes": 2,
  "views": 86,
  "uses": 22,
  "created_at": "2025-04-28T19:17:40Z",
  "title": "brand kit",
  "content": "brand kit what do i need",
  "category": "marketing",
  "subcategory": "branding",
  "tags": [
    "brand",
    "identity",
    "guidelines"
  ],
  "has_placeholders": false,
  "placeholders": [],
  "difficulty": "intermediate",
  "language": "en",
  "target_model": "gemini-2.0-flash"
}


In [3]:
from semantic_engine_demo.json_load import load_json
from pathlib import Path

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"

data_path = DATA_DIR / "dataset.json"

In [ ]:
import numpy as np

embs_np = np.load(DATA_DIR / "embeddings_02.npy").astype(np.float64)

embs_np[:5]

array([[-0.01766443,  0.08735338,  0.032757  , ..., -0.01891707,
        -0.05554084, -0.03803788],
       [-0.0469958 ,  0.00927899, -0.08253339, ..., -0.0500632 ,
         0.00752361,  0.06362955],
       [-0.003947  ,  0.00125577, -0.08268929, ...,  0.03811763,
        -0.03039739,  0.02121001],
       [-0.01960849, -0.0307398 , -0.01219587, ..., -0.07398218,
        -0.03380445,  0.02841577],
       [-0.04477432,  0.1259454 , -0.03261409, ..., -0.04117582,
        -0.04071708, -0.01830559]], shape=(5, 384))

In [5]:
from arrowspace import ArrowSpaceBuilder
import time

graph_params = {
    "eps": 0.8,
    "k": 25,
    "topk": 20,
    "p": 2.0,
    "sigma": None,
}

print(f"Graph parameters: {graph_params}")

print("Building ArrowSpace...")
start = time.perf_counter()
aspace, gl = (
    ArrowSpaceBuilder()
    .with_seed(42)
    .with_dims_reduction(enabled=False, eps=None)
    .with_sampling("simple", 1.0)
).build_and_store(graph_params, embs_np)
print(f"Build time: {time.perf_counter() - start:.2f}s")

Graph parameters: {'eps': 0.8, 'k': 25, 'topk': 20, 'p': 2.0, 'sigma': None}
Building ArrowSpace...
Build time: 26.52s


In [6]:
corpus = load_json(data_path)

In [7]:
import sqlite3

conn = sqlite3.connect('local_data.db')
cursor = conn.cursor()

In [8]:


def create_table(crs):
    
    # 2. Define and execute the table creation query
    # We'll create a sample 'users' table. 
    # IF NOT EXISTS prevents errors if you run this cell multiple times.
    create_table_query = '''
    CREATE TABLE IF NOT EXISTS docs_idx (
        id INTEGER PRIMARY KEY ,
        author_reputation INTEGER NOT NULL,
        version INTEGER NOT NULL,
        fork_count INTEGER NOT NULL,
        likes INTEGER NOT NULL,
        upvotes INTEGER NOT NULL,
        downvotes INTEGER NOT NULL,
        views INTEGER NOT NULL,
        uses INTEGER NOT NULL

    )
    '''
    crs.execute(create_table_query)

    # Commit the changes to the database
    

  

In [9]:
create_table(cursor)
conn.commit()
print("Database connected and 'users' table initialized.")

Database connected and 'users' table initialized.


In [10]:

insert_query = '''
INSERT INTO docs_idx (id, author_reputation, version, fork_count, likes, upvotes, downvotes, views, uses)
VALUES (:pos , :author_reputation, :version, :fork_count, :likes, :upvotes, :downvotes, :views, :uses)
'''

try:
    cursor.executemany(insert_query, corpus)

# 4. Commit to save the records to disk
    conn.commit()
except:
    pass

print(f"Successfully inserted {cursor.rowcount} records into the database.")

Successfully inserted -1 records into the database.


In [11]:
import pandas as pd

# Query the database and load it directly into a DataFrame
df = pd.read_sql_query("SELECT * FROM docs_idx WHERE id = 0", conn)

# Display the formatted table
display(df)

# Note: It's good practice to close the connection when you're completely finished
conn.close()

,id,author_reputation,version,fork_count,likes,upvotes,downvotes,views,uses
0,0,65,2,19,478,277,6,2770,776


In [12]:
import numpy as np

queries_path = DATA_DIR / "benchmark" / "queries_emb.npy"

queries_emb = np.load(queries_path)

In [13]:
result = aspace.search(queries_emb[2], gl, 0.8)

result

[(13080, 0.7525419747499589),
 (10373, 0.7501332062422266),
 (6672, 0.7220004884554212),
 (10829, 0.6951484474124662),
 (11983, 0.6940299371812412),
 (16635, 0.6886918862732456),
 (10738, 0.6871068049670491),
 (18171, 0.6772977474926476),
 (12480, 0.6709531254917455),
 (1978, 0.6673887044374611),
 (15094, 0.6627312625182755),
 (11197, 0.6568396383719872),
 (2071, 0.6549712070215256),
 (17680, 0.6433973251287074),
 (19710, 0.6432102702646942),
 (4037, 0.6430350328360734),
 (10921, 0.6394420208230435),
 (13700, 0.6342300906615181),
 (16345, 0.6297681445475828),
 (2604, 0.6297098109793728)]

In [14]:
queries_path_1 = DATA_DIR / "benchmark" / "benchmark_queries.json"
queries_corpus = load_json(queries_path_1)

In [15]:
queries_corpus[2]

{'entry_id': 'bq_0002',
 'source_pos': 13080,
 'source_id': 'pk_11090',
 'source_title': 'Design {{company_name}} A/B Test',
 'source_category': 'ab-testing',
 'source_subcategory': 'experiment-design',
 'source_uses': 268,
 'source_tags': ['experiment-design',
  'hypothesis-testing',
  'conversion-optimization',
  'statistical-power',
  'decision-framework'],
 'relevant_pos': {'tier1': [13080],
  'tier2': [2097, 9418, 10829, 13080, 13600, 14341, 16345]},
 'category_size': 7,
 'queries': {'title': 'Design {{company_name}} A/B Test',
  'expert': 'design company A/B test statistical power decision framework',
  'practitioner': 'design a/b test checklist for company revenue growth',
  'casual': 'design a/b test company',
  'novice': 'design a/b test for company experiment plan',
  'broken': 'design ab test company checklist statistical power',
  'cross_lingual': 'test design {company_name} A/B',
  'over_specified': 'ab testing design company A/B test expert level decision framework'}}

In [16]:
corpus[13080]

{'id': 'pk_11090',
 'author_reputation': 48,
 'version': 1,
 'fork_count': 27,
 'likes': 87,
 'upvotes': 76,
 'downvotes': 9,
 'views': 346,
 'uses': 268,
 'created_at': '2024-12-02T13:20:21Z',
 'title': 'Design {{company_name}} A/B Test',
 'content': "Design an A/B testing framework for {{company_name}}'s {{product_name}} to test whether changing the checkout button color from green to orange increases conversion rates. Include: (1) defining null and alternative hypotheses, (2) determining required sample size using power analysis, (3) random assignment strategy to ensure equal distribution, (4) metrics to track beyond primary conversion (bounce rate, cart abandonment, revenue per user), (5) how to handle multiple testing corrections if running secondary metric tests, and (6) decision framework for launch/reject based on results.",
 'category': 'ab-testing',
 'subcategory': 'experiment-design',
 'tags': ['experiment-design',
  'hypothesis-testing',
  'conversion-optimization',
  'stat

In [17]:
ids = [r[0] for r in result]

# 1. Fetch data safely
placeholders = ",".join(["?"] * len(ids))

case_clauses = " ".join([f"WHEN ? THEN {i}" for i in range(len(ids))])
order_query = f"ORDER BY CASE id {case_clauses} END"


query = f"SELECT * FROM docs_idx WHERE id IN ({placeholders}) {order_query}"

with sqlite3.connect('local_data.db') as conn:
    df = pd.read_sql_query(query, conn, params=ids + ids)

display(df)

,id,author_reputation,version,fork_count,likes,upvotes,downvotes,views,uses
0,13080,48,1,27,87,76,9,346,268
1,10373,25,1,96,442,220,28,2891,1569
2,6672,55,1,89,245,138,12,3235,1765
3,10829,65,1,6,84,62,4,389,54
4,11983,14,1,2,16,7,0,189,29
5,16635,61,2,9,83,13,1,452,39
6,10738,52,4,27,487,149,24,1637,978
7,18171,74,3,10,91,70,11,608,196
8,12480,76,2,10,157,100,17,2707,981
9,1978,20,3,1,45,45,1,341,78


In [ ]:
from semantic_engine_demo.benchmark_evaluation import extract_and_evaluate


predicted_results = result 
ground_truth_item = queries_corpus[2] 

# Run the evaluation
single_eval = extract_and_evaluate(
    predicted_results=predicted_results, 
    query_corpus_item=ground_truth_item, 
    k=10
)

print("Single Engine Evaluation:")
for key, value in single_eval.items():
    print(f"{key}: {value}")

Single Engine Evaluation:
entry_id: bq_0002
ndcg: 0.9027661662763876
tail_metrics: {'head_mean': np.float64(0.7415585564825355), 'tail_mean': np.float64(0.6598794974358021), 'tail_std': np.float64(0.022004347266003473), 'tail_to_head_ratio': np.float64(0.8898548761487359), 'tail_cv': np.float64(0.03334600840230563), 'tail_decay_rate': 0.0038493315548878432, 'n_tail_items': 17, 'total_items': 20}
